# Tavily Search API 발급 및 사용

## 실습 목표
---
Adaptive RAG를 구현하기 위해 Tavily Search API를 발급하고 LangChain을 통해 사용해 볼 것입니다.

24년 10월 기준 API의 무료 사용 가능 횟수는 월 1천회 로 제한되므로, 프로젝트에 사용할 분량은 남겨주세요.

## 실습 목차
---

1. **Tavily Search API 발급:** Tavily AI에 가입하고, API Key를 발급 받습니다.

2. **웹 검색 체인 구성:** 발급 받은 Tavily AI를 활용해 웹 데이터를 검색하고, 그 결과를 바탕으로 답변하는 체인을 구성합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [2]:
import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

load_dotenv()

True

- ChatGPT API(`.env`의 `MODEL_NAME`, 예: gpt-4o-mini)를 통해 모델을 불러옵니다.

## 1. Tavily Search API 발급

- Tavily Search API는 LLM과 RAG에 최적화된 웹 검색 API로써, LangChain 공식 튜토리얼에서도 활용하는 검색 API입니다.

1. 먼저, 아래 링크에 접속한 후 'Sign-in' 버튼을 눌러 로그인 화면으로 이동합니다.
   - https://app.tavily.com/sign-in
2. 그 다음, 여러분의 Google, GitHub 계정과 연동되는 계정을 만들거나 아래의 'Sign up' 버튼을 눌러 새로운 계정을 생성합니다.
3. 계정을 생성한 후, 아래 링크에 접속하여 default API Key를 복사합니다.
   - https://app.tavily.com/home
4. 이 노트북과 같은 폴더에 있는 `.env` 파일을 열어 `TAVILY_API_KEY=` 뒤에 복사한 키를 붙여넣고 저장합니다. (파일이 없다면 `.env.example`을 복사해서 `.env`로 이름을 바꾼 뒤 사용하세요.)

In [3]:
# API Key는 코드에 직접 입력하지 않고 .env 파일에서 불러옵니다.
# (.env 파일의 TAVILY_API_KEY에 키를 넣어두면 load_dotenv()가 자동으로 환경 변수에 등록합니다.)
assert os.environ.get("TAVILY_API_KEY"), ".env 파일에 TAVILY_API_KEY를 설정해주세요."

API Key를 성공적으로 등록했다면, 아래 코드를 실행해서 Tavily Search API가 잘 작동하는 지 확인합니다.

In [4]:
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(max_results=5)

In [5]:
tavily_search_tool.invoke({"query": "LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?"})

{'query': 'LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.ibm.com/kr-ko/think/topics/llamaindex-vs-langchain',
   'title': 'LlamaIndex 및 LangChain 비교: 차이점은 무엇인가요? | IBM',
   'content': '# LlamaIndex 및 LangChain 비교: 차이점은 무엇인가요? LlamaIndex와 LangChain은 검색 증강 생성(RAG) 시스템의 생성과 구현을 용이하게 하는 두 가지 플랫폼입니다. LlamaIndex는 간소화된 검색을 위해 구축되었으며 LangChain은 다양한 사용 사례를 지원하는 다목적 모듈식 플랫폼입니다. LlamaIndex는 텍스트 기반 데이터 소스에서 인덱싱, 데이터 수집 및 정보 검색에 중점을 두므로 보다 간단한 워크플로와 간단한 AI 애플리케이션에 이상적입니다. RAG는 생성형 AI 모델이 다른 방법으로는 부족할 수 있는 조직의 내부 데이터와 같은 도메인별 지식에 대해 엑세스할 수 있도록 지원합니다. 이전에 GPT Index로 알려졌던 LlamaIndex는 데이터 수집, 인덱싱 및 검색을 위한 오픈 소스 프레임워크입니다. LLAmainDex는 방대한 텍스트 데이터 세트를 쉽게 쿼리할 수 있는 인덱스로 변환하여 RAG 지원 콘텐츠 생성을 간소화합니다. #### 데이터 수집 및 LlamaHub. 데이터 수집은 LlamaIndex RAG 파이프라인의 첫 번째 단계입니다. 단일 데이터 소스에서 다루지 않는 검색 작업에 LlamaIndex를 사용하는 경우, 사용자는 다양한 특정 요구 사항을 충족하는 다목적 오픈 소스 데이터 로더 풀인 LlamaHub를 사용할 수 있습니다. LlamaIndex는 임베딩을 사용하여 사용자가 제공한 데이터를 검색 가능한 벡터

API 적용이 잘 되었다면, 이제 이 검색 결과를 활용해 답변을 생성하는 간단한 체인을 구성해 봅시다.

## 2. 웹 검색 체인 구성

웹 검색 기반 답변 체인은 RAG 기반 답변 체인과 비슷한 구조를 사용해 구현할 수 있습니다.

4챕터 실습3에서 저희는 Vector DB에서 사용자의 질문과 관련이 있는 Document를 Retrieve 했고, 그 텍스트를 이어 붙여서 프롬프트에 증강하였습니다.

이번 실습에서 구현할 웹 체인도 이와 비슷하게, 웹 검색 결과를 이어 붙여서 프롬프트에 증강합니다.


ChatGPT 모델을 사용하는 `ChatOpenAI` 객체를 생성합니다. `temperature=0`으로 설정해 같은 질문에 일관된 답을 유도합니다.

In [6]:
llm = ChatOpenAI(
    model=os.environ["MODEL_NAME"],
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)

다음으로, 사용자의 질문에 대해 Tavily Search API를 통해 여러 문서를 수집하고, 그 결과를 이어 붙이는 함수를 정의합니다.

In [7]:
def tavily_search_and_concat(query: str) -> str:
    # TavilySearch는 {"results": [{"content": ...}, ...]} 형태의 dict를 반환하므로,
    # 각 결과의 content만 뽑아 하나의 문자열로 이어 붙입니다.
    results = tavily_search_tool.invoke({"query": query})
    return "\n".join(result["content"] for result in results["results"])

In [8]:
print(tavily_search_and_concat("LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?")[:300])

# LlamaIndex 및 LangChain 비교: 차이점은 무엇인가요? LlamaIndex와 LangChain은 검색 증강 생성(RAG) 시스템의 생성과 구현을 용이하게 하는 두 가지 플랫폼입니다. LlamaIndex는 간소화된 검색을 위해 구축되었으며 LangChain은 다양한 사용 사례를 지원하는 다목적 모듈식 플랫폼입니다. LlamaIndex는 텍스트 기반 데이터 소스에서 인덱싱, 데이터 수집 및 정보 검색에 중점을 두므로 보다 간단한 워크플로와 간단한 AI 애플리케이션에 이상적입니다. RAG는 생성형 AI 모델이 다른 방법


`init_chain()` 함수를 정의합니다.
- 체인에서 `tavily_search_and_concat` 함수를 사용하는 것을 확인할 수 있습니다.

In [9]:
def init_chain():
    messages_with_contexts = [
        ("system", "웹 검색을 통해 수집한 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    # Context로 Tavily Serach API 결과를 이어 붙이는 함수를 사용합니다.
    qa_chain = (
        # 질문(question)은 그대로, context는 tavily_search_and_concat 결과로 구성합니다.
        {"context": tavily_search_and_concat, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )
    
    return qa_chain

In [10]:
qa_chain = init_chain()

Chain 구성이 완료되었으므로, 사용해봅시다.

In [11]:
question = "LangChain과 LlamaIndex의 특징과 차이점은 무엇인가요?"

print(qa_chain.invoke(question))

LlamaIndex와 LangChain은 모두 대형 언어 모델(LLM)을 기반으로 한 애플리케이션 개발을 위한 프레임워크이지만, 각각의 특징과 초점이 다릅니다.

### LlamaIndex
- **주요 기능**: LlamaIndex는 검색 및 정보 검색에 중점을 둔 프레임워크로, 데이터 인덱싱과 쿼리 처리에 강점을 가지고 있습니다. 대량의 텍스트 데이터를 효율적으로 처리하고 빠르고 정확한 정보 검색을 지원합니다.
- **사용 사례**: 주로 검색 증강 생성(RAG) 시스템을 구축하는 데 적합하며, 간단한 AI 애플리케이션이나 특정 데이터 소스에서의 검색 작업에 이상적입니다.
- **데이터 수집**: LlamaHub라는 오픈 소스 데이터 로더 풀을 통해 다양한 데이터 소스를 수집할 수 있으며, 사용자가 제공한 데이터를 벡터 기반 데이터 인덱스로 변환하여 검색 가능하게 합니다.
- **간소화된 워크플로**: LlamaIndex는 간단한 워크플로를 제공하여 사용자가 쉽게 인덱싱하고 검색할 수 있도록 돕습니다.

### LangChain
- **주요 기능**: LangChain은 모듈식이고 유연한 도구 세트를 제공하여 다양한 NLP 애플리케이션을 구축할 수 있도록 설계되었습니다. LLM 호출과 도구 사용 순서를 관리하는 데 강점을 가지고 있습니다.
- **사용 사례**: LLM에 국한되지 않고 다양한 애플리케이션을 지원하며, 데이터 로딩, 처리, 인덱싱 및 LLM과의 상호작용을 위한 도구를 제공합니다.
- **유연성**: 사용자가 데이터 세트의 특정 요구 사항에 맞게 애플리케이션을 맞춤화할 수 있는 기능을 제공합니다. 표준화된 인터페이스를 통해 프롬프트를 생성하고 관리할 수 있어 재사용이 용이합니다.
- **엔드 투 엔드 애플리케이션**: LangChain은 복잡한 LLM 애플리케이션을 구축하는 데 더 자주 사용되며, 체인과 에이전트 측면에서 더 완벽한 솔루션을 제공합니다.

### 결론
LlamaIndex는 검색 및 정보 검색에 최적화된 프레임워크로, 간단한 데이터

Tavily Search API를 활용해서 저희는 사용자의 질문에 대해 웹에서 정보를 수집해서 이를 바탕으로 답변하는 체인을 구성했습니다. 

Adaptive RAG를 구현하면서 이 체인을 활용해서 웹 검색 기능을 추가해 볼 것입니다.